# LASSI-Verify: Emperical Verification

**Argo Gateway API** Jupyter Notebook version

+ ***Post LASSI-Judge version with data collection for pass@k and energy_reduction@k***
+ No translation
+ Source code scripts from:
https://github.com/zjin-lcf/HeCBench/tree/master/src

NVIDIA GPU power measurements: https://pypi.org/project/nvidia-ml-py/  (pynvml)

Added support for AMD GPU device measurements and HIP code (ROCm)

In [1]:
from lassi.gpu_detector import detect_gpu_vendors

# Detect available GPUs
vendors = detect_gpu_vendors()
if not vendors:
    print("❌ No GPUs detected! LASSI requires GPU hardware.")
    raise SystemExit("Pipeline stopped - no GPU hardware found")

detected_vendor = vendors[0]['vendor']
gpu_count = vendors[0]['count']
print(f"🔍 Detected: {detected_vendor.upper()} with {gpu_count} GPU(s)")

# Vendor-specific initialization
if detected_vendor == "nvidia":
    # NVIDIA-specific setup
    import pynvml
    pynvml.nvmlInit()
    print("NVIDIA Driver Version:", pynvml.nvmlSystemGetDriverVersion())
    pynvml.nvmlShutdown()
    
    # Show NVIDIA system info
    !nvidia-smi
    !nvcc --version
    
elif detected_vendor == "amd":
    # AMD-specific setup
    from lassi.amd_power import amd_report_gpu_memory
    
    # Show AMD system info
    !rocminfo | head -20
    !rocm-smi
    !hipcc --version
    
print(f"\n✅ {detected_vendor.upper()} GPU environment ready for LASSI!")

🔍 Detected: NVIDIA with 1 GPU(s)
NVIDIA Driver Version: 570.211.01
Thu Mar 19 20:49:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX 5000 Ada Gene...    Off |   00000000:01:00.0 Off |                  Off |
| N/A   41C    P8              3W /   80W |      14MiB /  16376MiB |      0%      Default |
|                                         |                        |     

## Setup

In [2]:
import os
import json
import csv
import time
import shutil
from datetime import datetime
from zoneinfo import ZoneInfo

# Import LASSI modules
import lassi.llm_config
# import lassi.pipeline # The Pipeline script calls the other LASSI libraries not imported here.
import lassi.pipeline_v4
from lassi.format_colors import color
from lassi.prompt_dictionary import PromptDictionary
from lassi.monitor import start_monitor, log_figure

# Enable the autoreload extension
%load_ext autoreload
%reload_ext autoreload

# Reload all modules imported with %aimport every time before executing subsequent code.
%autoreload 1

# We need to reload the libraries while under development
%aimport lassi.format_colors
%aimport lassi.prompt_dictionary
%aimport lassi.llm_config
%aimport lassi.llm_methods
%aimport lassi.pipeline_v4
%aimport lassi.get_power
%aimport lassi.argostream_curl
%aimport lassi.code_manager
%aimport lassi.pipeline_methods
%aimport lassi.verify
%aimport lassi.monitor

In [3]:
# ── LASSI Monitor — live output dashboard ──────────────────────────────────────
# Starts a local web server and streams all pipeline print() output to a browser
# tab in real-time.  Run this cell once after the imports cell above.
# Then open the link below in a new browser tab and keep it visible.
#
# If matplotlib figures are produced (EE task), they appear automatically in a
# sidebar panel on the right side of the dashboard — no extra calls needed.
#
# The dashboard accumulates the full session history, so a late-opened tab
# will still see everything printed since start_monitor() was called.
#
# SSH remote access: forward port 5007 to your local machine, then open
#   http://localhost:5007  in your local browser.
start_monitor()   # default port 5007 — pass port=XXXX to change

'http://localhost:5007'

## Code Generation Experiment - Setup + Run

_____
***STEP 1/2: Input experimental configuration parameters in the following method.***

In [ ]:
def experimental_setup():
    # Load config file with source application names and input parameters
    with open("source_apps.json", "r") as f:
        source_apps = json.load(f)
    
    """
    [1] ENTER APPLICATION FOLDER NAME FROM HeCBench
        >> Execution input parameters provided.
    
    *** USE FOR LASSI-Verfify experiment: ***
    -- colorwheel ["10000", "256", "500000"] ["10000", "8", "1"]  # Usage: <range> <size> <repeat>
     (this one may show a lot of failures)

    HeCBench apps:
    -- matrix-rotate ["30000", "100"]  ["10000", "1"]  # Usage: %s <matrix size> <repeat>
    -- jacobi [""] (none required)
    -- layout ["2500"] ["1"]
    -- atomicCost ["16", "10"] ["1"]
    -- segment-reduce ["16384", "100"] (a replacement for atomicCost, same "Data compression and reduction" category in HeCBench)
       --> (may be trivial for LASSI-Verify)
    -- dense-embedding ["100000", "128", "100"] ["10000", "8", "1"]
    -- pathfinder ["100000", "10000", "2500"] ["100000", "1000", "5"] ["10000", "1000", "1000"]  # Usage: <column length> <row length> <pyramid_height>
    -- bsearch ["8388608", "1000"] ["10000", "1"]  # Usage: <number of elements> <repeat>
    -- entropy ["10000", "1024", "1"] ["8192", "8192", "500"]   # Usage: <width> <height> <repeat>
    -- colorwheel ["10000", "256", "500000"] ["10000", "8", "1"]  # Usage: <range> <size> <repeat>
    -- randomAccess ["10"] ["1"]  # Usage: <repeat>
    -- lid-driven-cavity [""]
    -- murmurhash3 ["750000, 500"]
    -- chacha20 ["300000"]
    -- jaccard ["1024, 512, 1000"]
    -- keogh ["256", "22500000", "100"]
    -- floydwarshall ["2048", "100", "16"]
    -- extrema ["750"]
    -- threadfence ["100", "250000000"]
    -- all-pairs-distance ["10000"]
    -- marchingCubes ["100"]

    Altis apps:
    -- level1/bfs ["--size 3"]
    -- level2/raytracing [""]
    -- level2/mandelbrot ["--imageSize 1024 --iterations 100000"]

    Miniapps:
    -- miniMDock ["-lfile", "./input/7cpa/7cpa_ligand.pdbqt", "-nrun 500"]
    -- XSBench ["-m", "event", "-l", "1700000000"]
    
    """

    # ****************************
    # ---> EXPERIMENT UPDATE FIELD
    
    # Enter source application to process through LASSI
    source_collection = "HeCBench"  # "Miniapps" "HeCBench"
    source_app = "entropy"  # "miniMDock" jacobi
    make_script = False  # Used only for Miniapps
    build_script = False  # False (for HeCBench) Use for Altis only
    

    """
    Valid source code folder names (under source_codes):
    1. HeCBench
    2. altis/src/cuda
    3. miniMDock/bin/
    """
    source_folder = source_apps[source_collection]["source_folder"]  # "altis/src/cuda/level2"    #  "HeCBench" "altis/src/cuda/level1"
    app_folder_name = source_apps[source_collection][source_app]["app_folder"]
    lines_of_code = source_apps[source_collection][source_app]["code_lines"]
    if make_script:
        app_name = source_app  # app_folder_name
        file_name = "Main"  # kernels
    else:
        app_name = "mt_" + app_folder_name + "-cuda_main"
        file_name = ""

    
    """
    [2] SELECT THE CODE TRANSLATION MAPPING DIRECTION EXPERIMENT CONFIGURATION
    
    CUDA -- OMP -- HIP -- SYCL -- FORTRAN
    """   
    # These should be the same if *not* doing translation.
    source_lang = "CUDA"
    target_lang = "CUDA"

    verifyprompt = True  # True or False -- Print out LLM prompt in stdout to preview
    
    # TOGGLE USE CODE KNOWLEDGE CONTEXT FOR TARGET
    use_context_summaries_only = True  # True or False
    few_shot_learning = False  # True or False
    
    # LLM judge code analysis (pre-cursor to LASSI-Verify)
    run_code_analysis = False  # True or False
    
    # THESE WILL NO LONGER BE OPTIONS
    # test_code_first = True  # True or False
    use_context = False  # True or False -- NOT USED IN v2, v3
    use_self_prompting = False  # True or False -- NOT USED IN v2, v3
    baseline_oneshot_goal = True  # True or False -- NOT USED IN v2, v3
    
    # -----------------------------------------------------------------------
    # TASK MODE CONFIGURATION
    # Set these two flags together depending on the task:
    #
    #   ENERGY EFFICIENCY (EE) task:
    #     power_measure    = True   -- measure GPU power/energy during execution
    #     yes_power_feedback = True -- run LASSI self-improvement loop using energy as the optimization metric
    #
    #   RUNTIME PERFORMANCE task:
    #     power_measure    = False  -- skip power measurement, use wall-clock duration instead
    #     yes_power_feedback = True -- run LASSI self-improvement loop using runtime as the optimization metric
    #
    #   CODE TRANSLATION task (one-shot, no iterative improvement):
    #     power_measure    = False  -- no measurement needed
    #     yes_power_feedback = False -- skip feedback loop entirely (generate once, compile, execute, judge, done)
    #     NOTE: translation mode still has a known infinite-loop-on-success issue when yes_power_feedback=False
    # -----------------------------------------------------------------------
    power_measure = False  # True (EE task) | False (runtime or translation task)
    vary_llm_temp = True  # True or False

    # TOGGLE LASSI SELF-IMPROVEMENT FEEDBACK LOOP
    yes_power_feedback = True  # True (EE or runtime task) | False (translation task, one-shot only)

    # -----------------------------------------------------------------------
    # LASSI-VERIFY TIER 2 CONFIGURATION
    # run_tier2=True: insert passthrough observation blocks and validate empirical equivalence
    # tier2_tolerance: relative floating-point tolerance for passthrough comparison (default 1e-6)
    # -----------------------------------------------------------------------
    run_tier2 = True         # True to enable Tier 2 empirical equivalence checking
    tier2_tolerance = 1e-6    # Relative tolerance for passthrough block comparison

    # LASSI-VERIFY TIER 3 CONFIGURATION
    # run_tier3=True: generate metamorphic contract and run variant filter (requires run_tier2=True)
    # tier3_tolerance: relative tolerance for metamorphic relation checks (default 1e-5, slightly looser than Tier 2)
    # -----------------------------------------------------------------------
    run_tier3 = True        # True to enable Tier 3 metamorphic testing
    tier3_tolerance = 1e-5   # Relative tolerance for metamorphic relation comparison

    # -----------------------------------------------------------------------
    # LLM-AS-A-JUDGE CONFIGURATION
    # run_llm_judge=True: enable LLM functional equivalence judge (pre-Tier-1/2 stop-gap)
    # run_llm_judge=False: pure three-tier mode — Tier 1 + Tier 2 determine validity only
    # -----------------------------------------------------------------------
    run_llm_judge = True     # True to enable LLM judge; False for pure three-tier LASSI-Verify mode

    # ASSIGN CODE COMPILE COMMAND
    code_build_script = ""
    code_compiler = ""
    code_compiler_kwds = ""
    if source_lang == "CUDA":
        if build_script:
            code_build_script = "build_" + app_name
        elif make_script:
            if source_app == "miniMDock":
                code_compiler = "make"
                code_compiler_kwds = "API=CUDA DEVICE=GPU CARD=NVIDIA NUMWI=64"
            elif source_app == "XSBench":
                code_compiler = "cuda/make"
        else:
            code_compiler = "nvcc"
            code_compiler_kwds = "-std=c++14 -Xcompiler -Wall -arch=sm_89 -O3"
            # -arch=sm_70 for aiops01 GPU (Argonne)  # -arch=sm_80 for A100s
            # -arch=sm_60 for P100
            # -arch=sm_89 for RTX 5000
        if yes_power_feedback:
            context_knowledge_selection = "cuda_optimization_summary"
            context_fewshot_examples = "cuda_c_few_shot_examples"
        else:
            context_knowledge_selection = "cuda"
    elif source_lang == "OMP":
        # For A100 on Linux_x86_64 or Linux_aarch64
        # code_compiler = "g++" # "/opt/nvidia/hpc_sdk/Linux_aarch64/2024/compilers/bin/nvc++"  # 
        # code_compiler_kwds = "-fopenmp -O3" # "-Wall -O3 -Minfo -mp=gpu -gpu=cc80"  # -c
        # !! TODO - REVIEW: code_compiler_kwds = "-Wall -O3 -Minfo=accel -mp=gpu -gpu=cc80"  # "-fopenmp -O3 -c"
        # For RTX 5000 Ada on Linux x86_64:
        code_compiler = "/opt/nvidia/hpc_sdk/Linux_x86_64/25.7/compilers/bin/nvc++" # "g++"
        code_compiler_kwds = "-Wall -O3 -Minfo=accel -mp=gpu -gpu=cc89"  # "-fopenmp -O3"
        context_knowledge_selection = "omp"
    elif source_lang == "HIP":
        if build_script:
            code_build_script = "build_" + app_name
        elif make_script:
            if source_app == "miniMDock":
                code_compiler = "make"
                code_compiler_kwds = "API=HIP DEVICE=GPU CARD=AMD NUMWI=64 HIP_PATH=/opt/rocm-6.3.3"
            elif source_app == "XSBench":
                code_compiler = "hip/make"
            else:
                code_compiler_kwds = ""
        else:
            code_compiler = "hipcc"
            code_compiler_kwds = "-std=c++14 -Wall -O3 --amdgpu-target=gfx908"
        if yes_power_feedback:
            context_knowledge_selection = "hip_optimization_summary"
            context_fewshot_examples = "hip_c_few_shot_examples"
        else:
            context_knowledge_selection = "hip"  # TODO

    # TOGGLE EXECUTE PIPELINE
    yes_execute = True  # True / False
    if yes_execute:
        # Enter execution input parameters, if needed
        # A list of parameter string values, e.g., ["10000", "1
        # ****************************
        # ---> EXPERIMENT UPDATE FIELD - Now, automated:
        exe_input_parameters = source_apps[source_collection][source_app]["app_params"]
    else:
        exe_input_parameters = [""]

    
    """
    [3] SELECT THE LLM TO USE FOR THE CODE TRANSLATION EXPERIMENT
    """
    # SELECT A MODEL

    # ****************************
    # ---> EXPERIMENT UPDATE FIELD
    
    # -- FROM ARGO:
    # model = "gpt35"  # (max token = 4096)
    # model = "gpt35large"  # (max token = 16384)
    # model = "gpt4large"  # (max token = 32768)
    # model = "gpt4turbo"  # (max token = 128000)
    # model = "gpto4mini"  # (max token = 128000)
    # model_coder = "gpto4mini"  # (max token = 128000)
    # model = "gpt4olatest"  # (max token = 128000)
    # model_coder = "gpt4olatest"  # (max token = 128000)
    # model_coder = "gpt4o"  # (max token = 128000)
    # model_coder = "gpt4olatest"  # (max token = 128000)
    # model = "gpt4olatest"  # (max token = 200000)
    # model_coder = "gpt4olatest"  # (max token = 200000)
    model = "gpt54"
    model_coder = "gpt54"
    model_judge = "gpt41"  # (max token = 200000)

    # -- FROM CEREBRAS
    # model = "gpt-oss-120b"
    # model_judge = "gpt-oss-120b"
    # model_coder = "qwen-3-coder-480b"
    
    # -- FROM OLLAMA
    # model = "devstral:24b" # (max token = 128000)
    # model_coder = "devstral:24b" # (max token = 128000)
    # model_judge = "devstral:24b" # (max token = 128000)
    # model = "qwq:latest"  # (max token = 128000)
    # model_coder = "qwen2.5-coder:32b"  # (max token = 32768)
    # model_judge = "qwq:latest"  # (max token = 128000)
    # model = "qwq:latest"  # (max token = 131072)
    # model = "gemma3:27b"  # crashing on code gen
    # model_coder = "gemma3:27b"
    # model = "granite3.2:8b-instruct-q8_0"
    # model_coder = "granite-code:34b-instruct-q8_0"
    # model = "deepseek-r1:70b"  # (max token = 128000)
    # model_coder = "deepseek-r1:70b"  # (max token = 128000)
    # model_coder = "deepseek-coder-v2:16b"  # (max token = 131072)
    # model_coder = "deepseek-coder-v2:16b"  # (max token = 131072)
    # model = "dolphin3"  # (max token = 131072)
    # model = "llama3.3:latest"  # (max token = 131072)
    # model = "phi4" # (max token = 16384)
    # model_coder = "codestral:22b"  # (max token = 32768)
    # model = "codestral:22b-v0.1-q8_0"  # (max token = 32768)
    # model = "wizardcoder:33b-v1.1-q8_0"  # (max token = 16384)
    # model_coder = "deepseek-coder:33b-instruct-q8_0"  # (max token = 16384)
    # model = "deepseek-coder-v2:16b-lite-instruct-fp16"  # (max token = 128000)
    # model = "deepseek-coder-v2:16b-lite-instruct-q8_0"
    # model = "phi3:14b"  # (max token = 131072)
    # model_judge = "llama3.3:70b"  # (max token = 128000)
    # model = "qwen3:latest"
    # model_coder = "qwen3:latest"
    # model_judge = "qwen3:latest"  # (max token = 40960)
    # model_coder = "qwen3-coder:30b"  # context 262144
    # model = "gpt-oss:20b"
    # model_judge = "gpt-oss:20b"
  
    # Set maximum context token count size for selected LLM:
    num_ctx = 200000
    num_ctx_coder = 200000

    # Needed only for Argo endpoints
    uri = "local"  # local  remote
    lassi.llm_config.argo_username = "mdearing"  # Argo Gateway username

    # ASSIGN LLM API SOURCE:
    if model in ["gpt54", "gpt5", "gpt35", "gpt35large", "gpt4", "gpt4large", "gpt4turbo", "gpt4o", "gpt4olatest", "gpt41",
                 "gpto1preview", "gpto1mini", "gpto3mini", "gpto4mini", "claudesonnet4", "claudesonnet46"]:
        llm_source = "argo"
    elif model in ["gpt-oss-120b", "qwen-3-coder-480b"]:
        llm_source = "cerebras"
    elif ("GGUF" in model and "hf.co" not in model):
        llm_source = "lmstudio"
    else:
        llm_source = "ollama"

    """
    [4] SET ADDITIONAL PIPELINE PARAMETERS
    """
    experiment = "TEST_FULLEXP_verify_0318-2026_" + model_coder + "_" + app_folder_name + "_runtm_" + detected_vendor.upper()  # Assign any label for experiment name.
    display_response = True
    test_with_error = False
    generate_test_case = False

    # Unload any previously loaded LLMs in the Ollama server 
    # (clear out the memory)
    if llm_source == "ollama":
        lassi.llm_config.unload_ollama_model(model)
        lassi.llm_config.unload_ollama_model(model_coder)

    """
    [5] SET DATA COLLECTION CONFIGURATION
    """
    # Data spreadsheet column headers
    DATAFIELDNAMES = [
        "trial_number",
        "timestamp",
        "experiment",
        "gpu_vendor",
        "app_name",
        "code_line_count",
        "llm",
        "llm_coder",
        "llm_judge",
        "source_v0_energy_J",
        "source_v0_avg_power_W",
        "source_v0_runtime_s",
        "source_verify_runtime_s",  # ★ instrumented source runtime
        "base_v1_energy_J",
        "base_v1_avg_power_W",
        "base_v1_runtime_s",
        "base_v1_valid",
        "base_v1_energy_diff",
        "base_v1_avg_power_diff",
        "base_v1_runtime_diff",
        "zeroshot_runtime_s",        # ★ schema-aligned alias for base_v1_runtime_s
        "zeroshot_compile_status",   # ★ "success" | "compile_error" | "skipped"
        "zeroshot_execute_status",   # ★ "success" | "execute_error" | "skipped"
        # "base_v1_optimization_methods",
        # "base_v2_energy_J",
        # "base_v2_avg_power_W",
        # "base_v2_runtime_s",
        # "base_v2_valid",
        # "base_v2_selfcorrections",
        # "base_v2_energy_diff",
        # "base_v2_avg_power_diff",
        # "base_v2_runtime_diff",
        # "base_v2_optimization_methods",
        "lassiee_energy_J",
        "lassiee_avg_power_W",
        "lassiee_runtime_s",
        "lassiee_valid",
        "lassiee_version",
        "lassiee_llm_temp",
        "lassiee_selfcorrections",
        "lassiee_eeiterations",
        "lassiee_energy_diff",
        "lassiee_avg_power_diff",
        "lassiee_runtime_diff",
        "lassiee_optimization_methods",
        "total_selfcorrections",
        "compile_selfcorrections",   # ★
        "execute_selfcorrections",   # ★
        "tier1_selfcorrections",     # ★
        "tier2_selfcorrections",     # ★
        "tier3_selfcorrections",     # ★
        "global_counter_at_end",     # ★ global verify counter at trial end
        "total_code_versions",       # ★ distinct LASSI code block indices
        "total_code_attempts",       # ★ rows in attempt log for this trial
        "total_eeiterations",
        "pipeline_runtime_min",
        "tier2_pass_count",
        "tier2_fail_total",
        "tier3_pass_count",
        "tier3_fail_total",
        "tier3_active_relations",
        "tier3_inactive_relations",  # ★
        "judge_false_positives",
        "judge_false_negatives",
        "lassiee_clean_status",
        "lassiee_clean_runtime_s",
        "lassiee_clean_runtime_diff",
        "lassiee_clean_energy_J",
        "lassiee_clean_energy_diff",
        "base_llm_tier1_verdict",
        "base_llm_tier2_verdict",
        "base_llm_tier3_verdict",
        "base_llm_verified_valid"
    ]
    
    return (app_folder_name, app_name, file_name, lines_of_code, source_lang, target_lang, model, model_coder, model_judge, num_ctx, num_ctx_coder, llm_source, use_context, use_context_summaries_only,
            few_shot_learning, run_code_analysis, context_knowledge_selection, source_folder, code_compiler, code_compiler_kwds, build_script, make_script, code_build_script, yes_execute, exe_input_parameters,
            experiment, display_response, test_with_error, generate_test_case, power_measure, yes_power_feedback, baseline_oneshot_goal, use_self_prompting, vary_llm_temp,
            verifyprompt, DATAFIELDNAMES, detected_vendor.upper(),
            run_tier2, tier2_tolerance, run_tier3, tier3_tolerance,
            run_llm_judge)

# EXPERIMENTAL DATA COLLETION MANAGEMENT
def init_data_collection(dir_path, DATAFIELDNAMES, filename="result_lassi-verify_pipeline_data.csv"):
    os.makedirs(dir_path, exist_ok=True)
    fp = os.path.join(dir_path, filename)
    if not os.path.exists(fp):
        with open(fp, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=DATAFIELDNAMES)
            writer.writeheader()
    return fp

_____
***STEP 2/2: Run the following block to implement the pipeline with the configuration parameters.***

In [5]:
# Experimental repeat for pass@k metrics
trial_number = 1
total_trials = 3

# Load prompt dictionary (to capture recent manual changes to script, if any)
prompts = PromptDictionary()

all_trials_start = time.time()

# Store the current home working directory (needed for code execution)
original_home_directory = os.getcwd()


# COLLECT THE EXPERIMENT CONFIGURATIONS
(
    app_folder_name, app_name, file_name, lines_of_code, source_lang, target_lang, model, model_coder, model_judge, num_ctx, num_ctx_coder,
    llm_source, use_context, use_context_summaries_only, few_shot_learning, run_code_analysis, context_knowledge_selection, source_folder, code_compiler,
    code_compiler_kwds, build_script, make_script, code_build_script, yes_execute, exe_input_parameters, experiment,
    display_response, test_with_error, generate_test_case, power_measure, yes_power_feedback, baseline_oneshot_goal,
    use_self_prompting, vary_llm_temp, verifyprompt, DATAFIELDNAMES, gpu_vendor,
    run_tier2, tier2_tolerance, run_tier3, tier3_tolerance,
    run_llm_judge
) = experimental_setup()

# IMPORT CONTEXT KNOWLEDGE FOR TARGET PROGRAMMING LANGUAGE
if gpu_vendor == "NVIDIA":
    input_context = "[BACKGROUND KNOWLEDGE: CUDA C++ PROGRAMMING BEST PRACTICES] " + prompts.get_codeknowledge("cuda") + "\n\n"  # [/END GENERAL CUDA PROGRAMMING CONTEXT]
    input_context = input_context + "[BACKGROUND KNOWLEDGE: CUDA C++ OPTIMIZATION GUIDE] " + prompts.get_codeknowledge(str(context_knowledge_selection)) + "\n\n"  #  [/END CUDA OPTIMIZATION CONTEXT]
elif gpu_vendor == "AMD":
    input_context = "[BACKGROUND KNOWLEDGE: HIP C++ PROGRAMMING BEST PRACTICES] " + prompts.get_codeknowledge("hip") + "\n\n"  # [/END GENERAL CUDA PROGRAMMING CONTEXT]
    input_context = input_context + "[BACKGROUND KNOWLEDGE: HIP C++ OPTIMIZATION GUIDE] " + prompts.get_codeknowledge(str(context_knowledge_selection)) + "\n\n"  #  [/END CUDA OPTIMIZATION CONTEXT]


# DEFINE PROMPT TEMPLATES
# TODO: MOST TASK PROMPT TEMPLATES WILL BE HANDLED BY THE PIPELINE METHODS
# If we are doing the runtime task, then set the system prompts, otherwise they are for ee
# we need to bring back in the translation systems prompt reference when we create LASSI-X
if (power_measure == False and yes_power_feedback == True):
    system_prompt_selection_index = "general_system_" + source_lang + "_runtime"
else:
    system_prompt_selection_index = source_lang + "_ee"  # "_to_" + target_lang  # Select which prompt to use from the prompt dictionary
this_llm_system_prompt = prompts.get_system_prompt(str(system_prompt_selection_index))

coderefactor_prompt_selection_index = source_lang  # + "_to_" + target_lang  # Select which prompt to use from the prompt dictionary
if (not baseline_oneshot_goal and use_self_prompting):
    # TODO: THIS SHOULD BE A DEAD BRANCH BECAUSE WE DON'T WANT TO USE baseline_oneshot_goal ANYMORE
    his_llm_refactoredcode_prompt = prompts.get_code_ee_refactoring_prompt(str(coderefactor_prompt_selection_index))
else: 
    if (power_measure == False and yes_power_feedback == True):
        # RUNTIME TASK
        if use_context:
            this_llm_refactoredcode_prompt = prompts.get_code_ee_refactoring_prompt(str(coderefactor_prompt_selection_index + "_baseline_runtime"))
        else:
            this_llm_refactoredcode_prompt = prompts.get_code_ee_refactoring_prompt(str(coderefactor_prompt_selection_index + "_baseline_nocontext_runtime"))
    else:
        # EE TASK
        if use_context:
            this_llm_refactoredcode_prompt = prompts.get_code_ee_refactoring_prompt(str(coderefactor_prompt_selection_index + "_baseline"))
        else:
            this_llm_refactoredcode_prompt = prompts.get_code_ee_refactoring_prompt(str(coderefactor_prompt_selection_index + "_baseline_nocontext"))

# LOAD INPUT CODE TO REFACTOR
input_code_folder = source_folder + "/" + app_folder_name + "/"
if build_script:
    input_code_filename = app_name
elif make_script:
    input_code_filename = file_name
else:
    input_code_filename = app_folder_name + "-" + source_lang.lower() + "_main"
if source_lang == "OMP" or source_lang == "SYCL":
    input_code_extension = "cpp"
elif source_lang == "CUDA" or source_lang == "HIP":
    if (make_script and source_lang == "HIP"):
        input_code_extension = "cpp"  # custom for miniMDock and XSBench
    else:
        input_code_extension = "cu"
if make_script:
    source_file_directory = "generated_codes/" + input_code_folder
else:
    source_file_directory = "source_codes/" + input_code_folder
if make_script:
    # SELECT FILE TO REFACTOR from miniapp
    if source_lang == "CUDA":
        if app_name == "miniMDock":
            input_code_file_subdirectory = "device/cuda/"
        elif app_name == "XSBench":
            input_code_file_subdirectory = ""
    elif source_lang == "HIP":
        if app_name == "miniMDock":
            input_code_file_subdirectory = "device/hip/"
        elif app_name == "XSBench":
            input_code_file_subdirectory = ""
    source_file_path = source_file_directory + input_code_file_subdirectory + input_code_filename + "." + input_code_extension
    source_file_path = os.path.abspath(source_file_path)
else:
    input_code_file_subdirectory = ""
    source_file_path = source_file_directory + input_code_filename + "." + input_code_extension
    source_file_path = os.path.abspath(source_file_path)
with open(source_file_path, 'r') as file:
    input_code = file.read()
file.close()


# SPECIFY OUTPUT CODE SAVE
if build_script:
    output_code_directory = "generated_codes/" + source_folder + "/" + app_folder_name
    output_code_file_name = app_name
elif make_script:
    output_code_file_name = file_name
    output_code_directory = "generated_codes/" + source_folder + "/" + app_folder_name
else:
    output_code_directory = "generated_codes/" + llm_source + "_" + model.replace('/', '--') + "/promptset_" + str(coderefactor_prompt_selection_index)
    output_code_file_name = input_code_filename + "_ee_refactorcode_" + source_lang + "_exp" + experiment

now = datetime.now(ZoneInfo("America/Chicago"))
dt_string = now.strftime("%m-%d-%Y_t%H_%M_%S")

# SPECIFIY DATA COLLECTION SAVE
experiment_data_directory = "generated_codes/experiment/" + experiment + "__" + dt_string
# Convert to absolute directory::
experiment_data_directory = os.path.abspath(experiment_data_directory)
experiment_data_file_object = init_data_collection(experiment_data_directory, DATAFIELDNAMES)
experiment_data = dict.fromkeys(DATAFIELDNAMES, None) # all keys start out as None
# Table 2 ★ per-attempt log — one row per code generation attempt (including self-correction retries)
attempt_log_file_path = init_data_collection(
    experiment_data_directory, lassi.pipeline_v4.ATTEMPT_LOG_FIELDNAMES,
    filename="result_lassi-verify_attempt_log.csv"
)
experiment_data["gpu_vendor"] = str(detected_vendor.upper())

while trial_number <= total_trials:
    # For the miniApp, we need to reset the source kernel file
    #  because the pipeline for this scenario must overwrite the 
    #  source code in the generated_codes folder.
    if make_script:
        if source_lang == "CUDA":
            if app_name == "miniMDock":
                working_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/miniMDock/device/cuda/kernels.cu"
                clean_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/miniMDock/device/cuda/kernels_orig_merged.cu"
            elif app_name == "XSBench":
                working_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/XSBench/cuda/Main.cu"
                clean_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/XSBench/cuda/Main_orig_merged.cu"
        elif source_lang == "HIP":
            if app_name == "miniMDock":
                working_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/miniMDock/device/hip/kernels.cpp"
                clean_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/miniMDock/device/hip/kernels_orig_merged.cpp"
            elif app_name == "XSBench":
                working_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/XSBench/hip/Main.cpp"
                clean_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/XSBench/hip/Main_orig_merged.cpp"
        try:
            print(f"[reset] Copying clean kernel -> working file\n"
                  f"        src: {clean_kernel_file}\n"
                  f"        dst: {working_kernel_file}")
            shutil.copyfile(clean_kernel_file, working_kernel_file)
            print("[reset] Success: working file reset to clean copy.")
        except FileNotFoundError as e:
            print(f"[reset] ERROR: file not found: {e.filename}")
        except PermissionError as e:
            print(f"[reset] ERROR: permission denied: {e.filename}")
        except OSError as e:
            # catches other IO errors (disk full, cross-device issues, etc.)
            print(f"[reset] ERROR: {e.__class__.__name__}: {e}")

    # CREATE METADATA RECORD CONTENT
    meta_data_file = "***** EXPERIMENT CONFIGURATION SUMMARY *****\n"
    meta_data_file += "\nGPU Vendor: " + str(detected_vendor.upper())
    meta_data_file += "\nDate-time: " + dt_string
    meta_data_file += "\nApplication: " + app_folder_name
    meta_data_file += "\n\nRefactoring task: " + system_prompt_selection_index
    meta_data_file += "\n\nLLM Source: " + llm_source
    meta_data_file += "\nModel: " + model
    meta_data_file += "\nModel coder: " + model_coder
    meta_data_file += "\nModel judge: " + model_judge
    meta_data_file += "\nSystem Prompt Index: " + str(coderefactor_prompt_selection_index)
    meta_data_file += "\nCode file template: " + output_code_directory + "/" + output_code_file_name
    meta_data_file += "\n\nLLM System Prompt: " + "\n\n" + this_llm_system_prompt
    # meta_data_file += "\n\nLLM Prompt for Code Generation: " + "\n\n" + this_llm_refactoredcode_prompt
    meta_data_file += "\n\nPower measurement: " + str(power_measure)
    
    # Display metadata
    print("\n" + meta_data_file + "\n")
    
    print("\n\n" + color.BLUE + color.BOLD + "***** BEGIN THE LASSI ENERGY EFFICIENCY PIPELINE (v4) *****" + color.END + " Trial #: " + str(trial_number) + "\n\n")
    
    # RUN LASSI-EE PIPELINE
    complete = lassi.pipeline_v4.auto_code_llmgeneration_pipeline(
        prompts,
        this_llm_system_prompt,
        this_llm_refactoredcode_prompt,
        llm_source, 
        model,
        model_coder,
        model_judge,
        num_ctx,
        num_ctx_coder,
        # arrayormatrix_size_range,
        # thread_count_range,
        use_context,
        use_context_summaries_only,
        few_shot_learning,
        run_code_analysis,
        input_context,
        input_code_file_subdirectory,
        input_code,
        lines_of_code,
        # target_code,
        target_lang,
        input_code_extension,
        source_lang,
        code_compiler,
        code_compiler_kwds,
        build_script,
        make_script,
        code_build_script,
        output_code_directory,
        output_code_file_name,
        source_file_path,
        display_response,
        test_with_error,
        generate_test_case,
        yes_execute,
        exe_input_parameters,
        meta_data_file,
        power_measure,
        yes_power_feedback,
        dt_string,
        baseline_oneshot_goal,
        use_self_prompting,
        vary_llm_temp,
        experiment,
        verifyprompt,
        trial_number,
        experiment_data_directory,
        experiment_data_file_object,
        experiment_data,
        app_name,
        run_tier2=run_tier2,
        tier2_tolerance=tier2_tolerance,
        run_tier3=run_tier3,
        tier3_tolerance=tier3_tolerance,
        run_llm_judge=run_llm_judge,
        attempt_log_file_path=attempt_log_file_path
    )
    
    # Reset to original working directory
    os.chdir(original_home_directory)
    
    if complete:
        print("\n\n" + color.GREEN + color.BOLD + "***** LASSI PIPELINE COMPLETED *****\n\n" + color.END)
    else:
        print("\n\n" + color.RED + color.BOLD + "***** LASSI PIPELINE FAILED TO COMPLETE *****\n\n" + color.END)

    # Increment experiment counter
    trial_number = trial_number + 1


all_trials_end = time.time() - all_trials_start

print("\n\n" + color.PURPLE + color.BOLD + "***** ALL TRIALS COMPLETED ***** Total time (min): " + str(round(all_trials_end/60, 2)) + "\n\n" + color.END)




***** EXPERIMENT CONFIGURATION SUMMARY *****

GPU Vendor: NVIDIA
Date-time: 03-19-2026_t20_49_51
Application: murmurhash3

Refactoring task: general_system_CUDA_runtime

LLM Source: argo
Model: gpt54
Model coder: gpt54
Model judge: gpt41
System Prompt Index: CUDA
Code file template: generated_codes/argo_gpt54/promptset_CUDA/murmurhash3-cuda_main_ee_refactorcode_CUDA_expTEST_FULLEXP_verify_0318-2026_gpt54_murmurhash3_runtm_NVIDIA

LLM System Prompt: 

You are an expert C++/CUDA engineer. Your goal is to refactor provided CUDA C++ code to reduce wall-clock execution time by improving GPU and CPU utilization and parallelism, while preserving all original functionality and expected output of the provided code. You must also carefully read and incorporate the best practices and CUDA optimization techniques from the provided background documents and refactoring plans. You always surround your generated code with the three characters ```.

Power measurement: False



***** BEGIN THE LASSI EN

Waiting for the system to settle down for 1 seconds...: 100%|██████████| 1/1 [00:01<00:00,  1.00s/it]


executable_file:  /home/mdearing/Research/LASSI/LASSI-Verify/source_codes/HeCBench/murmurhash3/murmurhash3-cuda_main
initial_dir:  /home/mdearing/Research/LASSI/LASSI-Verify
directory_path:  /home/mdearing/Research/LASSI/LASSI-Verify/source_codes/HeCBench/murmurhash3
executable:  murmurhash3-cuda_main
Working directory: /home/mdearing/Research/LASSI/LASSI-Verify/source_codes/HeCBench/murmurhash3
Execute command: ['./murmurhash3-cuda_main', '750000', '500']

running: executable -  murmurhash3-cuda_main

----- < BEGIN: STANDARD OUTPUT FROM CODE EXECUTION > -----

Average kernel execution time 0.031294 (s)
SUCCESS

THE RETURN CODE:  0

----- < END: Standard Output > -----


Program executed successfully!

----- END OF STAGE 0: TEST SOURCE -----

None

*** DEBUG: code_version: 0 | tracking_best_version: 0 | code_blocks len: 1 (next idx: 1) ***

--> ... prompt LLM to describe source code (Tier 1/2/3 context).


=====> VERIFY THE PROMPTING <===== [tokens: 2,578]

TEMPERATURE: 0.2

SYSTEM: Yo

```text
This program benchmarks and verifies a GPU implementation of MurmurHash3 x64 128-bit hashing across many variable-length keys.

Overall purpose
- It generates `numKeys` random-length byte arrays on the CPU.
- It computes a reference MurmurHash3 128-bit hash for each key on the CPU.
- It packs all keys into flat contiguous arrays suitable for GPU processing.
- It launches a CUDA kernel repeatedly, where each thread hashes one key.
- It measures average kernel execution time.
- It copies GPU results back and checks them against the CPU reference hashes.
- It prints either `SUCCESS` or `FAIL`.

Functional overview by block

1. Low-level MurmurHash3 helpers
These functions implement the core MurmurHash3 x64 128-bit algorithm and are marked for both host and device use so the same hashing logic runs on CPU and GPU.

- `rotl64(uint64_t x, int8_t r)`
  - Performs a 64-bit rotate-left operation.
  - Used heavily in MurmurHash mixing steps.

- `getblock64(const uint8_t* p, uint32_t i)`
  - Reads 8 bytes starting at block index `i` from a byte array and assembles them into a little-endian `uint64_t`.
  - Avoids assuming alignment.
  - Used to read 64-bit chunks from the input key.

- `fmix64(uint64_t k)`
  - Final avalanche mixing function from MurmurHash3.
  - Scrambles the bits of the hash state at the end to improve distribution.

- `MurmurHash3_x64_128(const void* key, uint32_t len, uint32_t seed, void* out)`
  - Full MurmurHash3 x64 128-bit implementation.
  - Inputs:
    - raw key pointer
    - key length in bytes
    - seed
    - output pointer for two 64-bit hash values
  - Behavior:
    - Processes the input in 16-byte blocks.
    - Handles any remaining tail bytes with a switch statement.
    - Mixes the total length into the state.
    - Applies finalization with `fmix64`.
    - Writes two 64-bit values to `out`.
  - Since it is `__host__ __device__`, it is reused for both CPU verification and GPU execution.

2. CUDA kernel
- `MurmurHash3_x64_128_kernel(...)`
  - GPU kernel where each thread hashes one key.
  - Parameters:
    - `d_keys`: flattened byte buffer containing all keys concatenated
    - `d_length`: prefix-offset array giving each key’s starting position in `d_keys`
    - `length`: per-key length array
    - `d_out`: output array holding two `uint64_t` values per key
    - `numKeys`: number of keys to process
  - For thread index `i`:
    - If `i < numKeys`, it hashes the key beginning at `d_keys + d_length[i]`
    - Uses `length[i]` as the byte count
    - Uses `i` itself as the hash seed
    - Stores the 128-bit result at `d_out + i*2`

Main data structures

On the host:
- `uint32_t* length`
  - Stores the length of each key.
  - Size: `numKeys`

- `uint8_t** keys`
  - Array of pointers, one per key.
  - Each key is separately allocated and filled with deterministic byte values.

- `uint64_t** out`
  - CPU reference output hashes.
  - Each entry holds two 64-bit words.

- `uint8_t* d_keys`
  - Host-side flattened buffer containing all key bytes concatenated.

- `uint32_t* d_length`
  - Host-side prefix sum / offset array.
  - `d_length[i]` is the start of key `i` in `d_keys`.
  - `d_length[i+1]` is the end offset.
  - Size: `numKeys + 1`

- `uint64_t* d_out`
  - Host-side flat output buffer used to receive hashes copied back from the GPU.
  - Size: `2 * numKeys`

On the device:
- `uint8_t* dev_keys`
  - Device copy of the flattened key bytes.

- `uint32_t* dev_length`
  - Device copy of the offset array for locating each key.

- `uint32_t* key_length`
  - Device copy of the per-key lengths.

- `uint64_t* dev_out`
  - Device output array for hashes.

Detailed main program flow

1. Argument parsing
- Expects exactly two command-line arguments:
  - number of keys
  - repeat count
- If not provided, prints usage and exits.

2. Host-side random test generation
- Seeds the C RNG with `srand(3)` for reproducibility.
- For each key:
  - Chooses a length randomly in `[0, 9999]`
  - Allocates a byte array of that length
  - Fills byte `c` with `c % 256`
  - Allocates two 64-bit output words
  - Computes CPU MurmurHash3 using seed `i`
- If `DEBUG` is enabled, prints each CPU-computed hash.

3. Packing keys for GPU processing
Because the GPU kernel cannot efficiently use an array of host pointers, the code converts the jagged key storage into flat arrays.

- Builds `d_length` as a prefix-sum offsets array:
  - `d_length[0] = 0`
  - `d_length[i+1] = d_length[i] + length[i]`
- Computes `total_length`, the total bytes across all keys.
- Allocates `d_keys` as one contiguous byte buffer of size `total_length`.
- Copies each `keys[i]` into `d_keys + d_length[i]`.
- Performs a sanity check using `memcmp` to ensure packing was correct.

4. GPU memory allocation and copies
- Allocates and populates:
  - `dev_keys` from `d_keys`
  - `dev_length` from `d_length`
  - `key_length` from `length`
- Allocates `dev_out` for the hash outputs.

5. Kernel launch configuration
- `BLOCK_SIZE` is fixed at 256.
- `blockDim` is set to 256 threads.
- `gridDim` is computed from `numKeys`, but notably as:
  - `((numKeys + BLOCK_SIZE - 1) / BLOCK_SIZE * BLOCK_SIZE)`
- This creates a 1D grid with a number of blocks equal to the rounded-up key count in units of 256, multiplied again by 256.
- Functionally, this still works because the kernel checks `if (i < numKeys)`, but it launches many more threads than necessary.

6. Timing and repeated execution
- Synchronizes before timing.
- Records start time using `std::chrono::steady_clock`.
- Launches the kernel `repeat` times.
- Synchronizes after the loop.
- Computes average kernel time and prints:
  - `Average kernel execution time ... (s)`

7. Copy back and verification
- Copies `dev_out` into host buffer `d_out`.
- Compares each GPU-computed hash pair against the CPU reference `out[i]`.
- Prints:
  - `FAIL` if any mismatch is found
  - `SUCCESS` otherwise

8. Cleanup
- Frees all host and device allocations.

Important functional characteristics

- Hash algorithm
  - Standard MurmurHash3 x64 128-bit.
  - Produces two 64-bit values per key.

- Seed usage
  - Each key uses its index `i` as the MurmurHash seed on both CPU and GPU.
  - This ensures the same input key data can still produce distinct hashes per key.

- Variable-length input handling
  - Keys can be zero-length up to 9999 bytes.
  - The implementation fully supports arbitrary lengths.

- Shared CPU/GPU logic
  - The same hashing function is compiled for host and device, reducing risk of algorithm mismatch between CPU reference and GPU implementation.

Major functions summary

- `rotl64`
  - 64-bit rotate-left utility.

- `getblock64`
  - Reads an unaligned little-endian 64-bit block from bytes.

- `fmix64`
  - Final avalanche mixing for MurmurHash3.

- `MurmurHash3_x64_128`
  - Complete 128-bit MurmurHash3 implementation for both CPU and GPU.

- `MurmurHash3_x64_128_kernel`
  - One-thread-per-key CUDA wrapper around the hash function.

- `main`
  - Drives test data generation, CPU reference hashing, key flattening, device transfers, kernel timing, correctness verification, and cleanup.

Notable implementation details and observations

- The code uses three length-related arrays/names:
  - `length`: host array of per-key lengths
  - `d_length`: host/device prefix offsets array
  - kernel parameter `length`: device array of per-key lengths
- The naming is somewhat confusing because `d_length` is not actually key lengths; it is offsets.

- The kernel accesses:
  - `d_keys + d_length[i]` as the start pointer
  - `length[i]` as the actual length

- The CPU and GPU outputs are stored differently:
  - CPU reference: one small allocation per key in `out[i]`
  - GPU result: flat array with two entries per key in `d_out`

- The code includes timing only for kernel execution, not for memory transfers or host-side preparation.

- The launch grid calculation is functionally safe but oversized, causing unnecessary threads and blocks.

In summary
This is a correctness-and-performance test harness for GPU-parallel MurmurHash3 hashing of many variable-length keys. It generates test inputs, computes trusted CPU hashes, packs inputs into flat buffers, hashes them in parallel on the GPU using one thread per key, times the kernel over repeated launches, and verifies exact output agreement with the CPU implementation.
```


----- STARTING CODE INSERT - Invariant Assertions -----


=====> VERIFY THE PROMPTING <===== [tokens: 19,751]

TEMPERATURE: 0.2

SYSTEM: You are an expert C++/CUDA engineer performing surgical code insertion. Your ONLY task is to INSERT new instrumentation code at specified locations in an existing source file. You MUST preserve ALL existing code EXACTLY as written — do not modify, simplify, reorder, optimize, or remove any existing code, comments, blank lines, or whitespace. Every single line of the input file must appear in your output, in the same order. You always surround your generated code with the three characters ```.

PROMPT: 
    You are a GPU code verification specialist.

    Your task is to instrument the provided CUDA C++ source code with **transformation-invariant
    assertion checks** — properties that must hold in the original code AND any correct refactored
    version, regardless of parallelization strategy, memory layout, or algorithmic optimization.

    These a

```cpp
//-------------------------------------------------------------------------
// MurmurHash3 was written by Austin Appleby, and is placed in the public
// domain. The author hereby disclaims copyright to this source code.
//-------------------------------------------------------------------------

#include <cstdlib>
#include <cstdint>
#include <cstdio>
#include <cstring>
#include <cassert>
#include <chrono>
#include <cuda.h>
#include <cmath>

#define BLOCK_SIZE 256

#define FORCE_INLINE inline __attribute__((always_inline))

__host__ __device__
inline uint64_t rotl64 ( uint64_t x, int8_t r )
{
  return (x << r) | (x >> (64 - r));
}

#define BIG_CONSTANT(x) (x##LU)

// Block read - if your platform needs to do endian-swapping or can only
// handle aligned reads, do the conversion here
__host__ __device__
FORCE_INLINE uint64_t getblock64 ( const uint8_t * p, uint32_t i )
{
  uint64_t s = 0;
  for (uint32_t n = 0; n < 8; n++) {
    s |= ((uint64_t)p[8*i+n] << (n*8));
  }
  return s;
}

// Finalization mix - force all bits of a hash block to avalanche
__host__ __device__
FORCE_INLINE uint64_t fmix64 ( uint64_t k )
{
  k ^= k >> 33;
  k *= BIG_CONSTANT(0xff51afd7ed558ccd);
  k ^= k >> 33;
  k *= BIG_CONSTANT(0xc4ceb9fe1a85ec53);
  k ^= k >> 33;
  return k;
}

__host__ __device__ 
void MurmurHash3_x64_128 (const void * key, const uint32_t len,
                          const uint32_t seed, void * out)
{
  const uint8_t * data = (const uint8_t*)key;
  const uint32_t nblocks = len / 16;

  uint64_t h1 = seed;
  uint64_t h2 = seed;

  const uint64_t c1 = BIG_CONSTANT(0x87c37b91114253d5);
  const uint64_t c2 = BIG_CONSTANT(0x4cf5ad432745937f);

  for(uint32_t i = 0; i < nblocks; i++)
  {
    uint64_t k1 = getblock64(data,i*2+0);
    uint64_t k2 = getblock64(data,i*2+1);

    k1 *= c1; k1  = rotl64(k1,31); k1 *= c2; h1 ^= k1;

    h1 = rotl64(h1,27); h1 += h2; h1 = h1*5+0x52dce729;

    k2 *= c2; k2  = rotl64(k2,33); k2 *= c1; h2 ^= k2;

    h2 = rotl64(h2,31); h2 += h1; h2 = h2*5+0x38495ab5;
  }

  const uint8_t * tail = (const uint8_t*)(data + nblocks*16);

  uint64_t k1 = 0;
  uint64_t k2 = 0;

  switch(len & 15)
  {
    case 15: k2 ^= ((uint64_t)tail[14]) << 48;
    case 14: k2 ^= ((uint64_t)tail[13]) << 40;
    case 13: k2 ^= ((uint64_t)tail[12]) << 32;
    case 12: k2 ^= ((uint64_t)tail[11]) << 24;
    case 11: k2 ^= ((uint64_t)tail[10]) << 16;
    case 10: k2 ^= ((uint64_t)tail[ 9]) << 8;
    case  9: k2 ^= ((uint64_t)tail[ 8]) << 0;
       k2 *= c2; k2  = rotl64(k2,33); k2 *= c1; h2 ^= k2;

    case  8: k1 ^= ((uint64_t)tail[ 7]) << 56;
    case  7: k1 ^= ((uint64_t)tail[ 6]) << 48;
    case  6: k1 ^= ((uint64_t)tail[ 5]) << 40;
    case  5: k1 ^= ((uint64_t)tail[ 4]) << 32;
    case  4: k1 ^= ((uint64_t)tail[ 3]) << 24;
    case  3: k1 ^= ((uint64_t)tail[ 2]) << 16;
    case  2: k1 ^= ((uint64_t)tail[ 1]) << 8;
    case  1: k1 ^= ((uint64_t)tail[ 0]) << 0;
       k1 *= c1; k1  = rotl64(k1,31); k1 *= c2; h1 ^= k1;
  };

  h1 ^= len; h2 ^= len;

  h1 += h2;
  h2 += h1;

  h1 = fmix64(h1);
  h2 = fmix64(h2);

  h1 += h2;
  h2 += h1;

  ((uint64_t*)out)[0] = h1;
  ((uint64_t*)out)[1] = h2;
}

__global__
void MurmurHash3_x64_128_kernel (
   const uint8_t *__restrict__ d_keys,
   const uint32_t *__restrict__ d_length,
   const uint32_t *__restrict__ length, 
         uint64_t *__restrict__ d_out,
   const uint32_t numKeys )
{
  uint32_t i = blockDim.x * blockIdx.x + threadIdx.x;
  if (i < numKeys) 
    MurmurHash3_x64_128 (d_keys+d_length[i], length[i], i, d_out+i*2);
}

int main(int argc, char** argv) 
{
  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate argc and basic parsed configuration at function entry
  {
    bool _t1_pass = (argc == 3);
    printf("LASSI-VERIFY ASSERTIONS [main_argc_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  if (argc != 3) {
    printf("Usage: %s <number of keys> <repeat>\n", argv[0]);
    return 1;
  } 
  uint32_t numKeys = atoi(argv[1]);
  uint32_t repeat = atoi(argv[2]);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate parsed problem sizes are positive
  {
    bool _t1_pass = (numKeys > 0 && repeat > 0);
    printf("LASSI-VERIFY ASSERTIONS [main_sizes_positive]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  srand(3);
  uint32_t i;
  // length of each key
  uint32_t* length = (uint32_t*) malloc (sizeof(uint32_t) * numKeys);
  // pointer to each key
  uint8_t** keys = (uint8_t**) malloc (sizeof(uint8_t*) * numKeys);
  // hashing output
  uint64_t** out = (uint64_t**) malloc (sizeof(uint64_t*) * numKeys);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate top-level host allocations for input metadata and reference outputs
  {
    bool _t1_pass = (length != NULL && keys != NULL && out != NULL);
    printf("LASSI-VERIFY ASSERTIONS [main_host_allocs_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  for (i = 0; i < numKeys; i++) {
    length[i] = rand() % 10000;
    keys[i] = (uint8_t*) malloc (length[i]);
    out[i] = (uint64_t*) malloc (2*sizeof(uint64_t));
    for (uint32_t c = 0; c < length[i]; c++) {
      keys[i][c] = c % 256;
    }
    MurmurHash3_x64_128 (keys[i], length[i], i, out[i]);
#ifdef DEBUG
    printf("%lu %lu\n", out[i][0], out[i][1]);
#endif
  }

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate generated key lengths and per-key host allocations after input generation
  {
    bool _t1_pass = true;
    for (uint32_t _t1_i = 0; _t1_i < numKeys; _t1_i++) {
      if (!(length[_t1_i] < 10000 && out[_t1_i] != NULL)) { _t1_pass = false; break; }
      if (length[_t1_i] > 0 && !(keys[_t1_i] != NULL)) { _t1_pass = false; break; }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_generated_inputs_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // create the 1D data arrays for device offloading  
  uint64_t* d_out = (uint64_t*) malloc (sizeof(uint64_t) * 2 * numKeys);
  uint32_t* d_length = (uint32_t*) malloc (sizeof(uint32_t) * (numKeys+1));

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate flattened host output and offset array allocations
  {
    bool _t1_pass = (d_out != NULL && d_length != NULL);
    printf("LASSI-VERIFY ASSERTIONS [main_flatten_allocs_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // initialize the length array
  uint32_t total_length = 0;
  d_length[0] = 0;
  for (uint32_t i = 0; i < numKeys; i++) {
    total_length += length[i];
    d_length[i+1] = total_length;
  }

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate prefix offsets are monotone and consistent with total packed length
  {
    bool _t1_pass = (d_length[0] == 0 && d_length[numKeys] == total_length);
    for (uint32_t _t1_i = 0; _t1_pass && _t1_i < numKeys; _t1_i++) {
      if (!(d_length[_t1_i] <= d_length[_t1_i+1] && d_length[_t1_i+1] - d_length[_t1_i] == length[_t1_i])) {
        _t1_pass = false;
        break;
      }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_offsets_consistent]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // initialize the key array 
  uint8_t* d_keys = (uint8_t*) malloc (sizeof(uint8_t) * total_length);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate flattened key storage allocation relative to total length
  {
    bool _t1_pass = ((total_length == 0) || (d_keys != NULL));
    printf("LASSI-VERIFY ASSERTIONS [main_flat_key_alloc_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  for (uint32_t i = 0; i < numKeys; i++) {
    memcpy(d_keys+d_length[i], keys[i], length[i]);
  }

  // sanity check
  for (uint32_t i = 0; i < numKeys; i++) {
    assert (0 == memcmp(d_keys+d_length[i], keys[i], length[i]));
  }

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate packed host key bytes match original per-key storage before host-to-device transfer
  {
    bool _t1_pass = true;
    for (uint32_t _t1_i = 0; _t1_i < numKeys; _t1_i++) {
      if (memcmp(d_keys+d_length[_t1_i], keys[_t1_i], length[_t1_i]) != 0) { _t1_pass = false; break; }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_htod_packed_keys_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate per-key byte pattern used to generate test inputs before host-to-device transfer
  {
    bool _t1_pass = true;
    for (uint32_t _t1_i = 0; _t1_i < numKeys && _t1_pass; _t1_i++) {
      for (uint32_t _t1_c = 0; _t1_c < length[_t1_i]; _t1_c++) {
        if (keys[_t1_i][_t1_c] != (uint8_t)(_t1_c % 256)) { _t1_pass = false; break; }
      }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_input_byte_pattern_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  uint8_t* dev_keys;
  cudaMalloc((void**)&dev_keys, sizeof(uint8_t)*total_length);
  cudaMemcpy(dev_keys, d_keys, sizeof(uint8_t)*total_length, cudaMemcpyHostToDevice);

  uint32_t* dev_length;
  cudaMalloc((void**)&dev_length, sizeof(uint32_t)*(numKeys+1));
  cudaMemcpy(dev_length, d_length, sizeof(uint32_t)*(numKeys+1), cudaMemcpyHostToDevice);

  uint32_t* key_length;
  cudaMalloc((void**)&key_length, sizeof(uint32_t)*(numKeys+1));
  cudaMemcpy(key_length, length, sizeof(uint32_t)*(numKeys), cudaMemcpyHostToDevice);

  uint64_t* dev_out;
  cudaMalloc((void**)&dev_out, sizeof(uint64_t)*(numKeys*2));

  dim3 gridDim ((numKeys+BLOCK_SIZE-1)/BLOCK_SIZE*BLOCK_SIZE);
  dim3 blockDim(BLOCK_SIZE);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate kernel launch configuration parameters
  {
    bool _t1_pass = (BLOCK_SIZE > 0 && BLOCK_SIZE <= 1024 && gridDim.x > 0 && blockDim.x > 0 && blockDim.x == BLOCK_SIZE);
    printf("LASSI-VERIFY ASSERTIONS [main_kernel_config_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  cudaDeviceSynchronize();
  auto start = std::chrono::steady_clock::now();

  for (uint32_t n = 0; n < repeat; n++)  
    MurmurHash3_x64_128_kernel<<<gridDim, blockDim>>>(
      dev_keys, dev_length, key_length, dev_out, numKeys);

  cudaDeviceSynchronize();
  auto end = std::chrono::steady_clock::now();
  auto time = std::chrono::duration_cast<std::chrono::nanoseconds>(end - start).count();
  printf("Average kernel execution time %f (s)\n", (time * 1e-9f) / repeat);

  cudaMemcpy(d_out, dev_out, sizeof(uint64_t)*(numKeys*2), cudaMemcpyDeviceToHost);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate copied-back hash outputs are finite-valued bit patterns when interpreted numerically
  {
    bool _t1_pass = true;
    for (uint32_t _t1_i = 0; _t1_i < 2 * numKeys; _t1_i++) {
      double _t1_as_double = (double)d_out[_t1_i];
      if (!std::isfinite(_t1_as_double)) { _t1_pass = false; break; }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_dtoh_output_finite_cast]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate copied-back GPU hashes exactly match CPU reference outputs after device-to-host transfer
  {
    bool _t1_pass = true;
    for (uint32_t _t1_i = 0; _t1_i < numKeys; _t1_i++) {
      if (!(d_out[2*_t1_i] == out[_t1_i][0] && d_out[2*_t1_i+1] == out[_t1_i][1])) { _t1_pass = false; break; }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_hash_outputs_match_reference]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // verify
  bool error = false;
  for (uint32_t i = 0; i < numKeys; i++) {
    if (d_out[2*i] != out[i][0] ||  d_out[2*i+1] != out[i][1]) {
      error = true;
      break;
    }
  }
  if (error) printf("FAIL\n");
  else printf("SUCCESS\n");

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate final success condition is consistent with exact host-side comparison result at function exit
  {
    bool _t1_pass = !error;
    printf("LASSI-VERIFY ASSERTIONS [main_final_verification_status]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - tier1_complete (sentinel — confirms all assertion blocks were reached)
  {
    printf("LASSI-VERIFY ASSERTIONS [tier1_complete]: PASS\n"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  for (uint32_t i = 0; i < numKeys; i++) {
    free(out[i]);
    free(keys[i]);
  }
  free(keys);
  free(out);
  free(length);
  free(d_keys);
  free(d_out);
  free(d_length);
  cudaFree(dev_keys);
  cudaFree(dev_out);
  cudaFree(dev_length);
  cudaFree(key_length);
  return 0;
}
```

... (281 outputs left out)

```cuda
//-------------------------------------------------------------------------
// MurmurHash3 was written by Austin Appleby, and is placed in the public
// domain. The author hereby disclaims copyright to this source code.
//-------------------------------------------------------------------------

#include <cstdlib>
#include <cstdint>
#include <cstdio>
#include <cstring>
#include <cassert>
#include <chrono>
#include <cuda.h>
#include <cuda_runtime.h>
#include <cmath>

#define BLOCK_SIZE 256
#define FORCE_INLINE inline __attribute__((always_inline))
#define CUDA_CHECK(call)                                                       \
  do {                                                                         \
    cudaError_t err__ = (call);                                                \
    if (err__ != cudaSuccess) {                                                \
      fprintf(stderr, "CUDA error at %s:%d: %s\n", __FILE__, __LINE__,         \
              cudaGetErrorString(err__));                                      \
      std::exit(EXIT_FAILURE);                                                 \
    }                                                                          \
  } while (0)

__host__ __device__
FORCE_INLINE uint64_t rotl64(uint64_t x, int8_t r)
{
  return (x << r) | (x >> (64 - r));
}

#define BIG_CONSTANT(x) (x##LU)

__host__ __device__
FORCE_INLINE uint64_t getblock64(const uint8_t* p, uint32_t i)
{
#if defined(__CUDA_ARCH__)
  const uint64_t* blocks = reinterpret_cast<const uint64_t*>(p);
  return blocks[i];
#else
  uint64_t s;
  std::memcpy(&s, p + 8 * i, sizeof(uint64_t));
  return s;
#endif
}

__host__ __device__
FORCE_INLINE uint64_t fmix64(uint64_t k)
{
  k ^= k >> 33;
  k *= BIG_CONSTANT(0xff51afd7ed558ccd);
  k ^= k >> 33;
  k *= BIG_CONSTANT(0xc4ceb9fe1a85ec53);
  k ^= k >> 33;
  return k;
}

__host__ __device__
FORCE_INLINE void MurmurHash3_x64_128(const void* key, const uint32_t len,
                                      const uint32_t seed, void* out)
{
  const uint8_t* data = static_cast<const uint8_t*>(key);
  const uint32_t nblocks = len / 16;

  uint64_t h1 = seed;
  uint64_t h2 = seed;

  const uint64_t c1 = BIG_CONSTANT(0x87c37b91114253d5);
  const uint64_t c2 = BIG_CONSTANT(0x4cf5ad432745937f);

  for (uint32_t i = 0; i < nblocks; i++) {
    uint64_t k1 = getblock64(data, i * 2 + 0);
    uint64_t k2 = getblock64(data, i * 2 + 1);

    k1 *= c1; k1 = rotl64(k1, 31); k1 *= c2; h1 ^= k1;
    h1 = rotl64(h1, 27); h1 += h2; h1 = h1 * 5 + 0x52dce729;

    k2 *= c2; k2 = rotl64(k2, 33); k2 *= c1; h2 ^= k2;
    h2 = rotl64(h2, 31); h2 += h1; h2 = h2 * 5 + 0x38495ab5;
  }

  const uint8_t* tail = data + nblocks * 16;

  uint64_t k1 = 0;
  uint64_t k2 = 0;

  switch (len & 15) {
    case 15: k2 ^= ((uint64_t)tail[14]) << 48;
    case 14: k2 ^= ((uint64_t)tail[13]) << 40;
    case 13: k2 ^= ((uint64_t)tail[12]) << 32;
    case 12: k2 ^= ((uint64_t)tail[11]) << 24;
    case 11: k2 ^= ((uint64_t)tail[10]) << 16;
    case 10: k2 ^= ((uint64_t)tail[ 9]) << 8;
    case  9: k2 ^= ((uint64_t)tail[ 8]) << 0;
             k2 *= c2; k2 = rotl64(k2, 33); k2 *= c1; h2 ^= k2;

    case  8: k1 ^= ((uint64_t)tail[ 7]) << 56;
    case  7: k1 ^= ((uint64_t)tail[ 6]) << 48;
    case  6: k1 ^= ((uint64_t)tail[ 5]) << 40;
    case  5: k1 ^= ((uint64_t)tail[ 4]) << 32;
    case  4: k1 ^= ((uint64_t)tail[ 3]) << 24;
    case  3: k1 ^= ((uint64_t)tail[ 2]) << 16;
    case  2: k1 ^= ((uint64_t)tail[ 1]) << 8;
    case  1: k1 ^= ((uint64_t)tail[ 0]) << 0;
             k1 *= c1; k1 = rotl64(k1, 31); k1 *= c2; h1 ^= k1;
  };

  h1 ^= len; h2 ^= len;
  h1 += h2;
  h2 += h1;

  h1 = fmix64(h1);
  h2 = fmix64(h2);

  h1 += h2;
  h2 += h1;

  reinterpret_cast<uint64_t*>(out)[0] = h1;
  reinterpret_cast<uint64_t*>(out)[1] = h2;
}

__global__
void MurmurHash3_x64_128_kernel(
    const uint8_t* __restrict__ d_keys,
    const uint32_t* __restrict__ d_offsets,
    const uint32_t* __restrict__ d_lengths,
    uint64_t* __restrict__ d_out,
    const uint32_t numKeys)
{
  const uint32_t i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < numKeys) {
    MurmurHash3_x64_128(d_keys + d_offsets[i], d_lengths[i], i, d_out + (size_t)i * 2);
  }
}

int main(int argc, char** argv)
{
  if (argc != 3) {
    printf("Usage: %s <number of keys> <repeat>\n", argv[0]);
    return 1;
  }

  const uint32_t numKeys = static_cast<uint32_t>(std::atoi(argv[1]));
  const uint32_t repeat  = static_cast<uint32_t>(std::atoi(argv[2]));

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate parsed input parameters at function entry
  {
    bool _t1_pass = (argv != NULL && numKeys > 0 && repeat > 0);
    printf("LASSI-VERIFY ASSERTIONS [main_entry_params_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  srand(3);

  uint32_t* length = static_cast<uint32_t*>(std::malloc(sizeof(uint32_t) * numKeys));
  uint8_t** keys   = static_cast<uint8_t**>(std::malloc(sizeof(uint8_t*) * numKeys));
  uint64_t** out   = static_cast<uint64_t**>(std::malloc(sizeof(uint64_t*) * numKeys));

  assert(length != nullptr || numKeys == 0);
  assert(keys != nullptr || numKeys == 0);
  assert(out != nullptr || numKeys == 0);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate primary host allocations
  {
    bool _t1_pass = (length != NULL && keys != NULL && out != NULL);
    printf("LASSI-VERIFY ASSERTIONS [main_host_allocs_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  for (uint32_t i = 0; i < numKeys; i++) {
    length[i] = rand() % 10000;
    keys[i] = static_cast<uint8_t*>(std::malloc(length[i] > 0 ? length[i] : 1));
    out[i] = static_cast<uint64_t*>(std::malloc(2 * sizeof(uint64_t)));
    assert(keys[i] != nullptr);
    assert(out[i] != nullptr);

    for (uint32_t c = 0; c < length[i]; c++) {
      keys[i][c] = static_cast<uint8_t>(c % 256);
    }

    MurmurHash3_x64_128(keys[i], length[i], i, out[i]);

#ifdef DEBUG
    printf("%lu %lu\n", out[i][0], out[i][1]);
#endif
  }

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate generated key metadata and CPU reference outputs
  {
    bool _t1_pass = true;
    for (uint32_t _t1_i = 0; _t1_i < numKeys; _t1_i++) {
      if (!(length[_t1_i] < 10000 && out[_t1_i] != NULL && (length[_t1_i] == 0 || keys[_t1_i] != NULL))) { _t1_pass = false; break; }
      if (!std::isfinite((double)out[_t1_i][0]) || !std::isfinite((double)out[_t1_i][1])) { _t1_pass = false; break; }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_generated_inputs_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  uint64_t* h_out_flat = static_cast<uint64_t*>(std::malloc(sizeof(uint64_t) * 2 * numKeys));
  uint32_t* h_offsets  = static_cast<uint32_t*>(std::malloc(sizeof(uint32_t) * (numKeys + 1)));
  uint8_t* h_keys_flat = nullptr;

  assert(h_out_flat != nullptr || numKeys == 0);
  assert(h_offsets != nullptr || numKeys == 0);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate packed host output and offset allocations
  {
    bool _t1_pass = (h_out_flat != NULL && h_offsets != NULL);
    printf("LASSI-VERIFY ASSERTIONS [main_packed_host_allocs_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  uint64_t total_length_64 = 0;
  h_offsets[0] = 0;
  for (uint32_t i = 0; i < numKeys; i++) {
    total_length_64 += length[i];
    assert(total_length_64 <= 0xFFFFFFFFull);
    h_offsets[i + 1] = static_cast<uint32_t>(total_length_64);
  }
  const uint32_t total_length = static_cast<uint32_t>(total_length_64);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate prefix offsets and total packed byte count
  {
    bool _t1_pass = (h_offsets[0] == 0 && h_offsets[numKeys] == total_length);
    if (_t1_pass) {
      for (uint32_t _t1_i = 0; _t1_i < numKeys; _t1_i++) {
        if (!(h_offsets[_t1_i] <= h_offsets[_t1_i+1] && h_offsets[_t1_i+1] - h_offsets[_t1_i] == length[_t1_i])) { _t1_pass = false; break; }
      }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_prefix_offsets_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  h_keys_flat = static_cast<uint8_t*>(std::malloc(total_length > 0 ? total_length : 1));
  assert(h_keys_flat != nullptr);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate flat key buffer allocation before packing
  {
    bool _t1_pass = (total_length == 0 || h_keys_flat != NULL);
    printf("LASSI-VERIFY ASSERTIONS [main_flat_key_buffer_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  for (uint32_t i = 0; i < numKeys; i++) {
    if (length[i] > 0) {
      std::memcpy(h_keys_flat + h_offsets[i], keys[i], length[i]);
    }
  }

  for (uint32_t i = 0; i < numKeys; i++) {
    assert(0 == std::memcmp(h_keys_flat + h_offsets[i], keys[i], length[i]));
  }

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate packed key bytes before host-to-device transfer
  {
    bool _t1_pass = true;
    for (uint32_t _t1_i = 0; _t1_i < numKeys; _t1_i++) {
      if (0 != std::memcmp(h_keys_flat + h_offsets[_t1_i], keys[_t1_i], length[_t1_i])) { _t1_pass = false; break; }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_packed_keys_match_original]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  CUDA_CHECK(cudaSetDevice(0));

  uint8_t* d_keys = nullptr;
  uint32_t* d_offsets = nullptr;
  uint32_t* d_lengths = nullptr;
  uint64_t* d_out = nullptr;

  CUDA_CHECK(cudaMalloc(reinterpret_cast<void**>(&d_keys), sizeof(uint8_t) * (total_length > 0 ? total_length : 1)));
  CUDA_CHECK(cudaMalloc(reinterpret_cast<void**>(&d_offsets), sizeof(uint32_t) * (numKeys + 1)));
  CUDA_CHECK(cudaMalloc(reinterpret_cast<void**>(&d_lengths), sizeof(uint32_t) * (numKeys > 0 ? numKeys : 1)));
  CUDA_CHECK(cudaMalloc(reinterpret_cast<void**>(&d_out), sizeof(uint64_t) * (numKeys > 0 ? (numKeys * 2ull) : 1ull)));

  CUDA_CHECK(cudaMemcpy(d_keys, h_keys_flat, sizeof(uint8_t) * total_length, cudaMemcpyHostToDevice));
  CUDA_CHECK(cudaMemcpy(d_offsets, h_offsets, sizeof(uint32_t) * (numKeys + 1), cudaMemcpyHostToDevice));
  CUDA_CHECK(cudaMemcpy(d_lengths, length, sizeof(uint32_t) * numKeys, cudaMemcpyHostToDevice));

  const dim3 blockDim(BLOCK_SIZE);
  const dim3 gridDim((numKeys + BLOCK_SIZE - 1) / BLOCK_SIZE);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate kernel launch configuration and device pointers
  {
    bool _t1_pass = (d_keys != NULL && d_offsets != NULL && d_lengths != NULL && d_out != NULL &&
                     blockDim.x > 0 && blockDim.x <= 1024 && gridDim.x > 0 &&
                     h_offsets[numKeys] == total_length);
    printf("LASSI-VERIFY ASSERTIONS [main_kernel_config_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  CUDA_CHECK(cudaDeviceSynchronize());
  auto start = std::chrono::steady_clock::now();

  for (uint32_t n = 0; n < repeat; n++) {
    MurmurHash3_x64_128_kernel<<<gridDim, blockDim>>>(d_keys, d_offsets, d_lengths, d_out, numKeys);
  }

  CUDA_CHECK(cudaGetLastError());
  CUDA_CHECK(cudaDeviceSynchronize());

  auto end = std::chrono::steady_clock::now();
  auto time = std::chrono::duration_cast<std::chrono::nanoseconds>(end - start).count();
  printf("Average kernel execution time %f (s)\n", (time * 1e-9f) / (repeat > 0 ? repeat : 1));

  // >>> BEGIN LASSI-VERIFY PASSTHROUGH [post_kernel_hash_output] (do not edit or delete this block) >>>
  {
    // --- LASSI-VERIFY PASSTHROUGH: capture kernel-written hash output buffer before host verification ---
    uint64_t* lv_host_buf = (uint64_t*) malloc((numKeys * 2) * sizeof(uint64_t));
    CUDA_CHECK(cudaMemcpy(lv_host_buf, d_out, (numKeys * 2) * sizeof(uint64_t), cudaMemcpyDeviceToHost));

    uint64_t lv_min = lv_host_buf[0], lv_max = lv_host_buf[0];
    double lv_sum = 0.0;
    for (int lv_i = 0; lv_i < (int)(numKeys * 2); lv_i++) {
        if (lv_host_buf[lv_i] < lv_min) lv_min = lv_host_buf[lv_i];
        if (lv_host_buf[lv_i] > lv_max) lv_max = lv_host_buf[lv_i];
        lv_sum += (double)lv_host_buf[lv_i];
    }
    double lv_mean = lv_sum / (numKeys * 2);

    printf("\n=== BEGIN LASSI-VERIFY PASSTHROUGH [post_kernel_hash_output] ===\n");
    printf("Buffer: dev_out\n");
    printf("Type: uint64_t\n");
    printf("Elements: %d\n", (int)(numKeys * 2));
    printf("Statistics: min=%.17g, max=%.17g, mean=%.17g\n",
           (double)lv_min, (double)lv_max, lv_mean);

    int lv_sample = ((numKeys * 2) < 10) ? (int)(numKeys * 2) : 10;
    printf("Sample_first10:");
    for (int lv_i = 0; lv_i < lv_sample; lv_i++)
        printf(" %.17g", (double)lv_host_buf[lv_i]);
    printf("\n");

    if ((numKeys * 2) > 10) {
        printf("Sample_last10:");
        for (int lv_i = (int)(numKeys * 2) - 10; lv_i < (int)(numKeys * 2); lv_i++)
            printf(" %.17g", (double)lv_host_buf[lv_i]);
        printf("\n");
    }

    if ((int)(numKeys * 2) > 10) {
        // Sample_spread10: 10 evenly-spaced interior elements at indices (k+1)*N/11.
        // Covers interior of the buffer regardless of array layout, catching bugs
        // that Sample_first10/last10 miss when boundary elements are unchanged.
        printf("Sample_spread10:");
        for (int lv_k = 0; lv_k < 10; lv_k++) {
            int lv_spread_idx = (lv_k + 1) * (int)(numKeys * 2) / 11;
            printf(" %.17g", (double)lv_host_buf[lv_spread_idx]);
        }
        printf("\n");
    }
    printf("=== END LASSI-VERIFY PASSTHROUGH [post_kernel_hash_output] ===\n");
    free(lv_host_buf);
  }
  // <<< END LASSI-VERIFY PASSTHROUGH <<<

  CUDA_CHECK(cudaMemcpy(h_out_flat, d_out, sizeof(uint64_t) * (numKeys * 2ull), cudaMemcpyDeviceToHost));

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate GPU output buffer after device-to-host transfer
  {
    bool _t1_pass = true;
    for (uint32_t _t1_i = 0; _t1_i < numKeys * 2; _t1_i++) {
      if (!std::isfinite((double)h_out_flat[_t1_i])) { _t1_pass = false; break; }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_gpu_output_finite]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // >>> BEGIN LASSI-VERIFY PASSTHROUGH [host_copied_hash_output] (do not edit or delete this block) >>>
  {
    // --- LASSI-VERIFY PASSTHROUGH: capture host-visible copied hash results used for verification ---
    uint64_t* lv_host_buf = (uint64_t*) malloc((numKeys * 2) * sizeof(uint64_t));
    for (int lv_i = 0; lv_i < (int)(numKeys * 2); lv_i++) {
        lv_host_buf[lv_i] = h_out_flat[lv_i];
    }

    uint64_t lv_min = lv_host_buf[0], lv_max = lv_host_buf[0];
    double lv_sum = 0.0;
    for (int lv_i = 0; lv_i < (int)(numKeys * 2); lv_i++) {
        if (lv_host_buf[lv_i] < lv_min) lv_min = lv_host_buf[lv_i];
        if (lv_host_buf[lv_i] > lv_max) lv_max = lv_host_buf[lv_i];
        lv_sum += (double)lv_host_buf[lv_i];
    }
    double lv_mean = lv_sum / (numKeys * 2);

    printf("\n=== BEGIN LASSI-VERIFY PASSTHROUGH [host_copied_hash_output] ===\n");
    printf("Buffer: d_out\n");
    printf("Type: uint64_t\n");
    printf("Elements: %d\n", (int)(numKeys * 2));
    printf("Statistics: min=%.17g, max=%.17g, mean=%.17g\n",
           (double)lv_min, (double)lv_max, lv_mean);

    int lv_sample = ((numKeys * 2) < 10) ? (int)(numKeys * 2) : 10;
    printf("Sample_first10:");
    for (int lv_i = 0; lv_i < lv_sample; lv_i++)
        printf(" %.17g", (double)lv_host_buf[lv_i]);
    printf("\n");

    if ((numKeys * 2) > 10) {
        printf("Sample_last10:");
        for (int lv_i = (int)(numKeys * 2) - 10; lv_i < (int)(numKeys * 2); lv_i++)
            printf(" %.17g", (double)lv_host_buf[lv_i]);
        printf("\n");
    }

    if ((int)(numKeys * 2) > 10) {
        // Sample_spread10: 10 evenly-spaced interior elements at indices (k+1)*N/11.
        // Covers interior of the buffer regardless of array layout, catching bugs
        // that Sample_first10/last10 miss when boundary elements are unchanged.
        printf("Sample_spread10:");
        for (int lv_k = 0; lv_k < 10; lv_k++) {
            int lv_spread_idx = (lv_k + 1) * (int)(numKeys * 2) / 11;
            printf(" %.17g", (double)lv_host_buf[lv_spread_idx]);
        }
        printf("\n");
    }
    printf("=== END LASSI-VERIFY PASSTHROUGH [host_copied_hash_output] ===\n");
    free(lv_host_buf);
  }
  // <<< END LASSI-VERIFY PASSTHROUGH <<<

  bool error = false;
  for (uint32_t i = 0; i < numKeys; i++) {
    if (h_out_flat[2ull * i] != out[i][0] || h_out_flat[2ull * i + 1] != out[i][1]) {
      error = true;
      break;
    }
  }

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate final GPU hashes match CPU reference
  {
    bool _t1_pass = (!error);
    printf("LASSI-VERIFY ASSERTIONS [main_final_hash_match]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  if (error) printf("FAIL\n");
  else printf("SUCCESS\n");

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - tier1_complete (sentinel — confirms all assertion blocks were reached)
  {
    printf("LASSI-VERIFY ASSERTIONS [tier1_complete]: PASS\n"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  for (uint32_t i = 0; i < numKeys; i++) {
    std::free(out[i]);
    std::free(keys[i]);
  }

  std::free(keys);
  std::free(out);
  std::free(length);
  std::free(h_keys_flat);
  std::free(h_out_flat);
  std::free(h_offsets);

  CUDA_CHECK(cudaFree(d_keys));
  CUDA_CHECK(cudaFree(d_offsets));
  CUDA_CHECK(cudaFree(d_lengths));
  CUDA_CHECK(cudaFree(d_out));

  return 0;
}
```

[Base LLM Verification] Compiling instrumented base LLM code ...
Compile command: ['nvcc', '-std=c++14', '-Xcompiler', '-Wall', '-arch=sm_89', '-O3', '/home/mdearing/Research/LASSI/LASSI-Verify/generated_codes/experiment/TEST_FULLEXP_verify_0318-2026_gpt54_murmurhash3_runtm_NVIDIA__03-19-2026_t20_49_51/murmurhash3-cuda_main_ee_refactorcode_CUDA_expTEST_FULLEXP_verify_0318-2026_gpt54_murmurhash3_runtm_NVIDIA_03-19-2026_t20_49_51_base_llm_verify.cu', '-o', '/home/mdearing/Research/LASSI/LASSI-Verify/generated_codes/experiment/TEST_FULLEXP_verify_0318-2026_gpt54_murmurhash3_runtm_NVIDIA__03-19-2026_t20_49_51/murmurhash3-cuda_main_ee_refactorcode_CUDA_expTEST_FULLEXP_verify_0318-2026_gpt54_murmurhash3_runtm_NVIDIA_03-19-2026_t20_49_51_base_llm_verify']

Compilation of code successful!

[Base LLM Verification] Executing ...
🔍 Using NVIDIA GPU for execution and power monitoring


Waiting for the system to settle down for 3 seconds...: 100%|██████████| 3/3 [00:03<00:00,  1.00s/it]


executable_file:  /home/mdearing/Research/LASSI/LASSI-Verify/generated_codes/experiment/TEST_FULLEXP_verify_0318-2026_gpt54_murmurhash3_runtm_NVIDIA__03-19-2026_t20_49_51/murmurhash3-cuda_main_ee_refactorcode_CUDA_expTEST_FULLEXP_verify_0318-2026_gpt54_murmurhash3_runtm_NVIDIA_03-19-2026_t20_49_51_base_llm_verify
initial_dir:  /home/mdearing/Research/LASSI/LASSI-Verify
directory_path:  /home/mdearing/Research/LASSI/LASSI-Verify/generated_codes/experiment/TEST_FULLEXP_verify_0318-2026_gpt54_murmurhash3_runtm_NVIDIA__03-19-2026_t20_49_51
executable:  murmurhash3-cuda_main_ee_refactorcode_CUDA_expTEST_FULLEXP_verify_0318-2026_gpt54_murmurhash3_runtm_NVIDIA_03-19-2026_t20_49_51_base_llm_verify
Working directory: /home/mdearing/Research/LASSI/LASSI-Verify/generated_codes/experiment/TEST_FULLEXP_verify_0318-2026_gpt54_murmurhash3_runtm_NVIDIA__03-19-2026_t20_49_51
Execute command: ['./murmurhash3-cuda_main_ee_refactorcode_CUDA_expTEST_FULLEXP_verify_0318-2026_gpt54_murmurhash3_runtm_NVIDIA_0